# Deep Dive: Agentic Workflow Fundamentals

## 📋 Summary

An **agent** is a small, named component that does *one job*: it takes an input, applies some logic
(plain code, an LLM call, or both) and returns an output.
An **agentic workflow** connects several agents so that **the output of one becomes the input of the next**.

Before we touch any LLM pattern (chaining, routing, parallelization...), we need the mental model behind all of them:

| Idea | In one sentence |
|---|---|
| Agent | A component with a single responsibility and a uniform `run()` interface |
| Workflow | The wiring: which agents run, in which order, and how data flows between them |
| Data contract | What each agent expects in and promises out (a string, a dict, a list...) |
| Flags / gates | Small signals an agent attaches to its output so the workflow can decide what happens next |
| Trace | A log of what each agent did, so you can debug the workflow |

### Workflow vs. agent: where this fits
| Approach | Who decides the next step? | Example |
|---|---|---|
| Single LLM call | Nobody, there is one step | "Summarize this ticket" |
| **Agentic workflow** (this course) | **You**, in code, ahead of time | intake → tag → reply |
| Autonomous agent | The model, at run time | "Resolve this ticket however you see fit" |

The patterns of this topic (chaining, routing, parallelization, evaluator-optimizer, orchestrator-workers) are all
ways to wire agents into a workflow, from the most predictable to the most dynamic.

### Key concepts in this notebook
| # | Concept | Complexity |
|---|---------|-----------|
| 1 | The smallest possible agent | 🟢 Beginner |
| 2 | Specialized agents through inheritance | 🟢 Beginner |
| 3 | A workflow is agents in sequence | 🟢 Beginner |
| 4 | Structured output, flags and branching | 🟡 Intermediate |
| 5 | A reusable `Workflow` class with a trace | 🟡 Intermediate |
| 6 | Mixing code agents and LLM agents | 🔴 Advanced |
| 7 | Exercises | 🏋️ Practice |

**How to use this notebook:** run the cells in order. Sections 1-5 are plain Python and need **no API key**;
only section 6 calls the OpenAI API. Cells marked `# TODO` are for you to fill in.

## 0. Setup

Sections 1-5 are plain Python and need **no API key**. The OpenAI client is set up at the start of section 6.

In [ ]:
import json
import os
import re
import time
from typing import Any, Dict, List

## 1. The Smallest Possible Agent 🟢

The base idea is tiny: **a name plus a `run()` method**. Every agent in this course, from a keyword checker to an
LLM-backed specialist, will expose the same `run()` so a workflow can call any of them without caring what is inside.

Our running example for sections 1-5 is a **customer-support pipeline** that turns a raw ticket into a reply draft.

In [ ]:
class Agent:
    """Base agent: a name and a run() method. Subclasses override run()."""

    def __init__(self, name: str):
        self.name = name

    def run(self, input_data: Any) -> Any:
        print(f"[{self.name}] received: {str(input_data)[:40]!r}")
        return input_data


echo = Agent("Echo")
result = echo.run("Refund my order #4411, the package never arrived!!")
print("returned:", result)

### 🧐 Observation
- The base agent returns its input untouched. Why is that a useful default for a base class?
- What would break if two agents used different method names (`run`, `process`, `execute`)?

## 2. Specialized Agents Through Inheritance 🟢

A real agent **overrides `run()`** with one focused responsibility. Notice that each agent below:
- does exactly one thing,
- declares what it takes in and what it returns (its *data contract*),
- has no idea which agents come before or after it.

In [ ]:
class IntakeAgent(Agent):
    """str -> dict. Cleans a raw ticket and extracts the order number."""

    def run(self, raw: str) -> Dict:
        text = " ".join(raw.split())                      # collapse whitespace
        match = re.search(r"#(\d+)", text)
        return {"text": text, "order_id": match.group(1) if match else None}


class TaggerAgent(Agent):
    """dict -> dict. Adds topic tags based on keywords."""

    KEYWORDS = {
        "billing":  ["refund", "charged", "invoice"],
        "shipping": ["package", "delivery", "tracking"],
        "urgent":   ["asap", "immediately", "today", "!!"],
    }

    def run(self, ticket: Dict) -> Dict:
        lower = ticket["text"].lower()
        tags = [tag for tag, words in self.KEYWORDS.items() if any(w in lower for w in words)]
        return {**ticket, "tags": tags}


class ReplyDraftAgent(Agent):
    """dict -> str. Drafts a reply from the tags (rule-based for now)."""

    def run(self, ticket: Dict) -> str:
        topics = " and ".join(t for t in ticket["tags"] if t != "urgent") or "your request"
        order = f" (order #{ticket['order_id']})" if ticket["order_id"] else ""
        return f"Hi! Thanks for reaching out about {topics}{order}. We're looking into it and will reply within 24h."


intake, tagger, drafter = IntakeAgent("Intake"), TaggerAgent("Tagger"), ReplyDraftAgent("Reply Drafter")

step1 = intake.run("   Refund my order   #4411,  the package never arrived!!  ")
print(step1)

## 3. A Workflow Is Agents in Sequence 🟢

The simplest workflow is a **pipeline**: feed the initial input to the first agent, then feed each output to the next agent.
This is exactly what you do by hand below. Notice how **information flows** and **each agent transforms the data**.

In [ ]:
raw_ticket = "   Refund my order   #4411,  the package never arrived!!  "

cleaned = intake.run(raw_ticket)
print("  →", cleaned)

tagged = tagger.run(cleaned)
print("  →", tagged)

reply = drafter.run(tagged)
print("  →", reply)

### 🧐 Observation
- Which agent would you change to also extract an email address? Do the other two need to change?
- What happens if you swap the order of `tagger` and `intake`? What error do you get, and why is it a *data contract* problem?

## 4. Structured Output, Flags and Branching 🟡

Returning a **dict** instead of a plain string lets an agent attach **extra signals** next to the main content
(`tags`, `flags`, a confidence, a count...). The workflow can then **branch** on those signals instead of blindly
running every step.

Here, tickets tagged `urgent` skip the automatic reply and go to a human escalation queue.

In [ ]:
class EscalationAgent(Agent):
    """dict -> dict. Pushes the ticket to a human queue instead of auto-replying."""

    def run(self, ticket: Dict) -> Dict:
        return {"queue": "human-tier-2", "reason": "urgent tag", "ticket": ticket}


escalator = EscalationAgent("Escalation")


def handle_ticket(raw: str) -> Any:
    ticket = tagger.run(intake.run(raw))
    if "urgent" in ticket["tags"]:
        print("  ⚠️  Flag: urgent → escalating")
        return escalator.run(ticket)
    print("  ✅ No urgent flag → auto-reply")
    return drafter.run(ticket)


for raw in [
    "Where is my package? Order #7 tracking has not moved.",
    "I was charged twice for order #88, fix this ASAP!!",
]:
    print("\nTicket:", raw)
    print("  →", handle_ticket(raw))

### 🧐 Observation
- The two branches return **different types** (`str` vs `dict`). What problem does that cause for whoever consumes the result? How would you fix it?
- The keyword tagger is cheap but brittle ("not urgent" would still match `urgent`). When would you replace it with an LLM agent?

## 5. A Reusable `Workflow` Class with a Trace 🟡

Hand-wiring steps works for one pipeline. A **general-purpose** workflow treats the step list as **data**:
give it any list of agents and it runs them in order and **records a trace** (who ran, how long, what type came out).
This is the mindset of the course project: a reusable toolkit of agents plus a workflow that assembles them.

In [ ]:
class Workflow:
    """Runs agents in order, passing each output to the next agent, and records a trace."""

    def __init__(self, name: str, steps: List[Agent]):
        self.name = name
        self.steps = steps
        self.trace: List[Dict] = []

    def run(self, data: Any) -> Any:
        self.trace = []
        print(f"🚀 Starting workflow '{self.name}' with {len(self.steps)} steps")
        for agent in self.steps:
            start = time.perf_counter()
            data = agent.run(data)
            self.trace.append({
                "agent": agent.name,
                "seconds": round(time.perf_counter() - start, 4),
                "output_type": type(data).__name__,
            })
        print(f"✅ Workflow '{self.name}' completed")
        return data


support_flow = Workflow("Support triage", [intake, tagger, drafter])
final = support_flow.run("My delivery #52 is late, tracking shows nothing.")
print("\nFinal:", final)
print("\nTrace:")
for row in support_flow.trace:
    print(" ", row)

### 🧐 Observation
- The `Workflow` has no idea what the agents do. What makes that a strength?
- What would you add to the trace to debug a wrong answer three steps in (hint: the *input* and *output* of each step)?

## 6. Mixing Code Agents and LLM Agents 🔴

Because every agent honours the same `run()` contract, **an LLM-backed agent is a drop-in replacement** for a rule-based one.
Use **code for what is deterministic** (cleaning, keyword checks, routing on a flag) and **an LLM for what needs language
understanding** (tone, summarizing, drafting).

Below, `LLMAgent` wraps a system prompt behind the same interface, and we swap it in for the rule-based `ReplyDraftAgent`.

### API setup (only needed from here on)

In [ ]:
# Key input: in Colab, add OPENAI_API_KEY in the 🔑 Secrets panel (left sidebar).
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from getpass import getpass
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [ ]:
from openai import OpenAI
from IPython.display import Markdown, display

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = "gpt-4.1-nano"  # cheap model, plenty for this experiment


def chat(system: str, user: str, temperature: float = 0.3) -> str:
    """Single-turn helper: returns the assistant text."""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=temperature,
    )
    return resp.choices[0].message.content


def show(label: str, text: str):
    display(Markdown(f"**{label}**\n\n{text}\n\n---"))

In [ ]:
class LLMAgent(Agent):
    """Any input -> str. Same run() contract, but the logic is a prompt."""

    def __init__(self, name: str, system_prompt: str, temperature: float = 0.3):
        super().__init__(name)
        self.system_prompt = system_prompt
        self.temperature = temperature

    def run(self, input_data: Any) -> str:
        user = json.dumps(input_data, indent=2) if isinstance(input_data, dict) else str(input_data)
        return chat(self.system_prompt, user, self.temperature)


llm_drafter = LLMAgent(
    "LLM Reply Drafter",
    system_prompt=(
        "You are a support agent. You receive a JSON ticket with 'text', 'order_id' and 'tags'. "
        "Write a warm, specific reply in under 60 words. Mention the order id if there is one. "
        "Never promise a refund."
    ),
)

hybrid_flow = Workflow("Support triage (hybrid)", [intake, tagger, llm_drafter])
answer = hybrid_flow.run("I was charged twice for order #88 and nobody answers my emails.")
show("Hybrid workflow reply", answer)
print(hybrid_flow.trace)

### 🧐 Observation
- Compare this reply with the rule-based one. What did the LLM add, and what risk did it introduce?
- Which two agents stayed as plain code, and why is it a good idea to keep them that way?

## 7. Exercises 🏋️

### Exercise 1: Add a new agent to the pipeline
Add a `PriorityAgent` between `TaggerAgent` and the reply step. It should read `tags`, add a `priority` key
(`"high"` if `urgent` is tagged, `"medium"` if `billing` is tagged, otherwise `"low"`), and the reply drafter should
mention the priority. Run it through `Workflow` and print the trace.

In [ ]:
class PriorityAgent(Agent):
    """dict -> dict. Adds a 'priority' key based on the tags."""

    def run(self, ticket: Dict) -> Dict:
        # TODO: compute the priority from ticket["tags"] and return a new dict
        pass


# TODO: build a Workflow([...]) that uses intake, tagger, PriorityAgent and a drafter that mentions the priority
# TODO: run it on a ticket and print the final reply and the trace

### Exercise 2: Conditional steps
Right now `Workflow` always runs every step. Extend it so a step can be **skipped** based on a condition
(for example "only run `EscalationAgent` when `urgent` is in the tags").

Design question first: how will you express the condition, and what should the trace record for a skipped step?

In [ ]:
class ConditionalWorkflow(Workflow):
    """Workflow where each step is (agent, condition); the step only runs if condition(data) is True."""

    def __init__(self, name: str, steps: List[tuple]):
        super().__init__(name, [agent for agent, _ in steps])
        self.conditions = [cond for _, cond in steps]

    def run(self, data: Any) -> Any:
        self.trace = []
        # TODO: loop over steps and conditions, skip a step when its condition is False,
        #       and record 'skipped' rows in self.trace
        return data

### Exercise 3: Design a hybrid workflow
Pick a small task from your own work or studies (for example "turn meeting notes into action items").
1. List the stages and mark which are **code** and which need an **LLM**.
2. Define the input and output type of each stage.
3. Implement at least two stages with the `Agent` / `LLMAgent` classes above.

In [ ]:
# TODO: describe your task and its stages
stages = [
    {"name": "Stage 1", "kind": "code", "input": "str", "output": "dict"},
    # add more stages
]

# TODO: implement at least two of the stages as Agent subclasses and run them in a Workflow

## 🏁 Summary & Key Takeaways

- An **agent** = one responsibility + a uniform `run()` interface.
- A **workflow** = agents wired in a defined order, where data flows from step to step.
- **Data contracts** (what goes in, what comes out) are what make agents replaceable.
- Structured outputs with **flags** let the workflow branch instead of running blindly.
- Use **code for deterministic work** and **an LLM for language work**: both fit the same interface.
- A **trace** turns "it gave a weird answer" into "step 3 received the wrong input".

### What's next?
- **Deep Dive 2: Prompt Chaining with Agents**: wire LLM agents in a chain where each agent's output feeds the next prompt.